# CUDA MatrixLab: Colab benchmark runner

This notebook compiles and runs the **real** CUDA and C++ programs from `cuda/` on a Google Colab NVIDIA GPU,
verifies every result against a CPU reference, and exports the measurements to
`cuda_matrixlab_results.json`. Import that file into the local CUDA MatrixLab UI.

**Before you start**
1. *Runtime → Change runtime type → Hardware accelerator: **T4 GPU*** (or any GPU runtime).
2. Upload `CUDA-MatrixLab.zip` when the upload cell asks for it.
3. *Runtime → Run all*. The benchmark cell can take several minutes; the CPU baseline at n=2048 is the slowest part.

Nothing in this notebook is hard-coded. Every number printed comes from a program run in this session.

In [ ]:
# 1) Confirm that an NVIDIA GPU is visible.
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
# 2) Upload CUDA-MatrixLab.zip and extract it to /content/CUDA-MatrixLab.
import os, zipfile
from google.colab import files

uploaded = files.upload()
zip_name = next((n for n in uploaded if n.lower().endswith(".zip")), None)
assert zip_name, "Please upload CUDA-MatrixLab.zip"

with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content")

PROJECT = "/content/CUDA-MatrixLab"
CUDA = os.path.join(PROJECT, "cuda")
REQUIRED = ["common.h", "common_cuda.h", "vector_add.cu", "matrix_cpu.cpp",
            "matrix_cuda_naive.cu", "matrix_cuda_shared.cu"]
missing = [f for f in REQUIRED if not os.path.isfile(os.path.join(CUDA, f))]
assert not missing, f"Missing files in the zip: {missing}"
print("Project extracted to", PROJECT)

In [ ]:
# 3) Read the GPU, CUDA toolkit and CPU details. These go into the results file.
import re, subprocess, platform, datetime as dt

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

def gpu_query(fields):
    r = sh(f"nvidia-smi --query-gpu={fields} --format=csv,noheader,nounits")
    if r.returncode != 0 or not r.stdout.strip():
        return None
    return [x.strip() for x in r.stdout.strip().splitlines()[0].split(",")]

info = gpu_query("name,driver_version")
assert info, "No NVIDIA GPU detected. Set Runtime > Change runtime type > T4 GPU, then rerun."
gpu_name, driver_version = info
cc = gpu_query("compute_cap")
compute_cap = cc[0] if cc else None
arch_flag = f"-arch=sm_{compute_cap.replace('.', '')}" if compute_cap else "-arch=native"

nvcc_text = sh("nvcc --version").stdout
m = re.search(r"release (\d+\.\d+)", nvcc_text)
cuda_version = m.group(1) if m else None
assert cuda_version, "nvcc not found. Colab normally provides it; try Runtime > Restart session."

cpu_model = platform.processor() or "unknown"
with open("/proc/cpuinfo") as f:
    for line in f:
        if line.startswith("model name"):
            cpu_model = line.split(":", 1)[1].strip()
            break

print(f"GPU: {gpu_name} | driver {driver_version} | compute capability {compute_cap}")
print(f"CUDA toolkit: {cuda_version} | arch flag: {arch_flag}")
print(f"CPU: {cpu_model}")

In [ ]:
# 4) Compile the four programs. CPU uses g++; CUDA programs use nvcc.
BUILD = os.path.join(PROJECT, "build")
os.makedirs(BUILD, exist_ok=True)

PROGRAMS = {
    "cpu_matmul":         ("matrix_cpu.cpp",         "cpu"),
    "cuda_naive_matmul":  ("matrix_cuda_naive.cu",   "cuda"),
    "cuda_shared_matmul": ("matrix_cuda_shared.cu",  "cuda"),
    "cuda_vector_add":    ("vector_add.cu",          "cuda"),
}

BINS = {}
for kernel, (src, kind) in PROGRAMS.items():
    out = os.path.join(BUILD, os.path.splitext(src)[0])
    if kind == "cuda":
        cmd = ["nvcc", "-O3", "-std=c++17", arch_flag, "-I", CUDA, os.path.join(CUDA, src), "-o", out]
    else:
        cmd = ["g++", "-O3", "-std=c++17", "-I", CUDA, os.path.join(CUDA, src), "-o", out]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError(f"Compilation failed: {src}")
    BINS[kernel] = out
    print("built", os.path.basename(out))

In [ ]:
# 5) Run every benchmark. Each program prints one JSON line on stdout.
import json, time

def run_program(binary, n, reps, timeout=3600):
    t0 = time.time()
    r = subprocess.run([binary, str(n), str(reps)], capture_output=True, text=True, timeout=timeout)
    if r.returncode != 0:
        raise RuntimeError(f"{os.path.basename(binary)} failed at n={n}:\n{r.stderr}")
    line = [l for l in r.stdout.splitlines() if l.strip()][-1]
    rec = json.loads(line)
    print(f"{rec['kernel']:<20} n={rec['n']:<10} median={rec['median_ms']:.4f} ms "
          f"verified={rec['verified']} (wall {time.time() - t0:.1f} s)")
    return rec

PLAN = [
    ("cpu_matmul",         [256, 512, 1024, 2048],        3),
    ("cuda_naive_matmul",  [256, 512, 1024, 2048, 4096],  10),
    ("cuda_shared_matmul", [256, 512, 1024, 2048, 4096],  10),
    ("cuda_vector_add",    [2**20, 2**22, 2**24, 2**26],  20),
]

runs = []
for kernel, sizes, reps in PLAN:
    for n in sizes:
        runs.append(run_program(BINS[kernel], n, reps))
print(f"\nFinished {len(runs)} runs.")

In [ ]:
# 6) Stop here if any result failed verification.
bad = [r for r in runs if not r["verified"]]
assert not bad, f"Verification failed for {len(bad)} run(s): {bad}"
print(f"All {len(runs)} runs verified against the double-precision CPU reference.")

In [ ]:
# 7) Write the results file and download it.
doc = {
    "schema": "cuda-matrixlab-results/v1",
    "generated_at": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "source": "google-colab",
    "environment": {
        "os": platform.platform(),
        "python": platform.python_version(),
        "cpu_model": cpu_model,
        "gpu_name": gpu_name,
        "driver_version": driver_version,
        "compute_capability": compute_cap,
        "cuda_version": cuda_version,
        "compiler": sh("g++ --version").stdout.splitlines()[0],
        "nvcc_arch_flag": arch_flag,
    },
    "runs": runs,
}
OUT = "/content/cuda_matrixlab_results.json"
with open(OUT, "w") as f:
    json.dump(doc, f, indent=2)
print("Saved", OUT, f"({len(runs)} runs)")
files.download(OUT)

In [ ]:
# 8) Optional: quick table and chart of the matrix-multiplication results.
import matplotlib.pyplot as plt

by_n = {}
for r in runs:
    if r["kernel"] != "cuda_vector_add":
        by_n.setdefault(r["n"], {})[r["kernel"]] = r["median_ms"]

print(f"{'n':>6} {'CPU ms':>12} {'naive ms':>12} {'shared ms':>12} {'shared vs naive':>16}")
for n in sorted(by_n):
    d = by_n[n]
    ratio = d["cuda_naive_matmul"] / d["cuda_shared_matmul"] if "cuda_shared_matmul" in d else float("nan")
    print(f"{n:>6} {d.get('cpu_matmul', float('nan')):>12.3f} {d['cuda_naive_matmul']:>12.3f} "
          f"{d['cuda_shared_matmul']:>12.3f} {ratio:>15.2f}x")

sizes = sorted(by_n)
plt.figure(figsize=(8, 4.5))
for key, label in [("cpu_matmul", "CPU"), ("cuda_naive_matmul", "CUDA naive"), ("cuda_shared_matmul", "CUDA shared")]:
    xs = [n for n in sizes if key in by_n[n]]
    plt.plot(xs, [by_n[n][key] for n in xs], marker="o", label=label)
plt.xscale("log", base=2); plt.yscale("log")
plt.xlabel("matrix size n (n x n)"); plt.ylabel("median time (ms, log scale)")
plt.title("Measured matrix multiplication times"); plt.grid(True, which="both", alpha=0.3); plt.legend()
plt.tight_layout(); plt.show()